In [1]:
import pickle

movies = pickle.load(open("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/models/movies.pkl", "rb"))
content_similarity = pickle.load(open("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/models/similarity.pkl", "rb"))
collaborative_similarity = pickle.load(open("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/models/collaborative_similarity.pkl", "rb"))

In [11]:
def recommend(movie):

    movie_index = movies[movies["title"] == movie].index[0]

    distances = content_similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )

    recommendations = []

    for i in movie_list[1:6]:

        recommendations.append(
            (
                movies.iloc[i[0]].title,
                i[1]
            )
        )

    return recommendations

In [13]:
print(recommend("Avatar"))

[('Aliens vs Predator: Requiem', np.float64(0.29061909685954823)), ('Aliens', np.float64(0.2726248784031354)), ('Falcon Rising', np.float64(0.26401000024165)), ('Independence Day', np.float64(0.25903973506580724)), ('Titan A.E.', np.float64(0.2537477434955704))]


In [6]:
def recommend_movie(movie):

    if movie not in collaborative_similarity.columns:
        return []

    similar_movies = collaborative_similarity[movie]

    similar_movies = similar_movies.sort_values(
        ascending=False
    )

    recommendations = []

    for movie_name, score in similar_movies.iloc[1:6].items():

        recommendations.append(
            (
                movie_name,
                score
            )
        )

    return recommendations

In [15]:
recommend_movie("Toy Story (1995)")

[('Toy Story 2 (1999)', 0.5726012603197153),
 ('Jurassic Park (1993)', 0.5656368040861564),
 ('Independence Day (a.k.a. ID4) (1996)', 0.5642616935276659),
 ('Star Wars: Episode IV - A New Hope (1977)', 0.5573881705799365),
 ('Forrest Gump (1994)', 0.5470959079401742)]

In [17]:
def hybrid_recommend(movie):

    # Get recommendations from both models
    content = recommend(movie)
    collaborative = recommend_movie(movie)

    # Dictionary to store final scores
    final_scores = {}

    # Give 60% weight to Content-Based recommendations
    for movie_name, score in content:
        final_scores[movie_name] = score * 0.6

    # Give 40% weight to Collaborative recommendations
    for movie_name, score in collaborative:

        if movie_name in final_scores:
            final_scores[movie_name] += score * 0.4

        else:
            final_scores[movie_name] = score * 0.4

    # Check the combined scores
    print(final_scores)

In [18]:
hybrid_recommend("Avatar")

{'Aliens vs Predator: Requiem': np.float64(0.17437145811572893), 'Aliens': np.float64(0.16357492704188123), 'Falcon Rising': np.float64(0.15840600014498998), 'Independence Day': np.float64(0.15542384103948434), 'Titan A.E.': np.float64(0.15224864609734223)}


In [19]:
def hybrid_recommend(movie):

    content = recommend(movie)

    collaborative = recommend_movie(movie)

    final_scores = {}

    # Content-Based Weight = 60%
    for movie_name, score in content:

        final_scores[movie_name] = score * 0.6

    # Collaborative Weight = 40%
    for movie_name, score in collaborative:

        if movie_name in final_scores:

            final_scores[movie_name] += score * 0.4

        else:

            final_scores[movie_name] = score * 0.4

    # Sort by Final Score
    sorted_movies = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended_movies = []

    for movie_name, score in sorted_movies[:5]:

        recommended_movies.append(movie_name)

    return recommended_movies

In [21]:
hybrid_recommend("Iron Man 2")

['Krrish',
 'Ant-Man',
 'The Animal',
 'Iron Man 3',
 'The Adventures of Elmo in Grouchland']